# Chapter 4: Polars — High-Performance DataFrames

### Why This Matters

Pandas was designed in 2008, single-threaded by default, and holds entire datasets in memory using patterns that, at multi-gigabyte scale, become genuinely slow. **Polars**, written in Rust and released publicly in 2020, was designed from scratch to use every CPU core in parallel and to optimize your entire sequence of operations *before* running any of them — often 5–50x faster than equivalent Pandas code on large datasets, with a smaller memory footprint. You don't need it for a 10,000-row sales file; you'll want it the day you're handed a 10-million-row transaction log and Pandas starts taking minutes instead of seconds.

### 4.1 Eager Mode — Looks Like Pandas, Runs Differently

In [ ]:
import polars as pl

df = pl.DataFrame({"region": ["East", "West", "East", "South"], "sales": [100, 120, 180, 90]})
print(df.group_by("region").agg(pl.col("sales").sum().alias("total_sales")))

The overall shape is deliberately familiar if you know Pandas — you build a DataFrame, group by a column, and aggregate. The syntax differs in one important way: Polars uses **expressions** — `pl.col("sales").sum()` — rather than the string-based `.agg({"sales": "sum"})` style Pandas uses. An expression is a small, composable object describing a computation (here: "take the sales column, sum it") that Polars can inspect, optimize, and combine with other expressions *before* actually executing anything.

### 4.2 Lazy Mode — Query Planning Before Execution

In [ ]:
lf = (
    df.lazy()
    .filter(pl.col("sales") > 100)
    .group_by("region")
    .agg(pl.col("sales").mean().alias("avg_sales"))
)
print(lf.explain())    # shows the OPTIMIZED query plan — inspect this before running on large data
print(lf.collect())    # actually EXECUTES the plan and returns a real DataFrame

This is the single biggest conceptual difference from Pandas. `.lazy()` doesn't run anything — it builds up a description of the work you want done (filter, then group, then aggregate). Only `.collect()` actually executes it. In between, Polars' query optimizer inspects the whole plan and can, for example, push the `filter` step earlier or reorder operations to touch less data overall — the same kind of optimization a SQL database's query planner does before running your query, applied here to DataFrame operations. `.explain()` lets you literally read that optimized plan before committing to running it, which is invaluable when you're about to point a pipeline at a genuinely large file and want to sanity-check what will actually happen.

💡 **Practical guidance:** for day-to-day exploratory analysis (small-to-medium data, fast iteration), eager mode (no `.lazy()`) is simpler and perfectly adequate — the overhead of query planning isn't worth it. Reach for `.lazy()` specifically when you're building a pipeline you intend to run repeatedly against large files, or reading directly from a large file (`pl.scan_csv(...)` reads lazily, meaning it doesn't even load the file into memory until `.collect()` is called, and can skip columns/rows the final query doesn't need).

### Cheat Sheet — Polars

| Pandas | Polars equivalent |
|---|---|
| `pd.DataFrame(dict)` | `pl.DataFrame(dict)` |
| `df.groupby("x")["y"].sum()` | `df.group_by("x").agg(pl.col("y").sum())` |
| `df[df["x"] > 5]` | `df.filter(pl.col("x") > 5)` |
| `df["new"] = df["x"] * 2` | `df.with_columns((pl.col("x") * 2).alias("new"))` |
| `pd.read_csv(path)` | `pl.read_csv(path)` (eager) or `pl.scan_csv(path)` (lazy) |
| — | `.lazy() ... .collect()` (query planning — no Pandas equivalent) |

### 🎯 Practice — Polars

In [ ]:
import polars as pl

# Rewriting a Pandas groupby as a Polars lazy pipeline
lf = (
    pl.DataFrame({
        "region": ["North","South","East","West"]*3,
        "sales": [125,98,112,145,130,101,120,150,99,140,133,152]
    })
    .lazy()
    .filter(pl.col("sales") > 100)
    .with_columns((pl.col("sales") * 1.1).alias("projected_sales"))
    .group_by("region")
    .agg([
        pl.col("sales").sum().alias("total_sales"),
        pl.col("projected_sales").mean().alias("avg_projected")
    ])
    .sort("total_sales", descending=True)
)
print(lf.collect())

⚠️ Polars is a fast-moving library — pin a specific version in any `requirements.txt` for production work, since method names and defaults have shifted between releases more than in the more mature Pandas/NumPy ecosystem.

---

## Volume 02 — What You Should Be Able to Do Now

- Explain why NumPy arrays are faster than Python lists, and what "vectorization" means concretely
- Filter both a NumPy array and a Pandas DataFrame using boolean conditions, correctly using `&`/`|`
- Load a real CSV, inspect it with `.info()`/`.describe()`, clean missing values, and justify your cleaning choice
- Use `.groupby().agg()` to answer a "total/average by category" business question, and explain the split-apply-combine idea in your own words
- Convert data between long and wide format with `.melt()` / `.pivot_table()`
- Explain, in one sentence, when you'd reach for Polars instead of Pandas

**Next: `03_Data_Visualization.md`**